# Phase 4 & 5 — Historical Simulation VaR + EWMA Volatility Scaling
Runs the var/ and volatility/ modules and displays results. All logic lives in `src/`, this notebook only calls it and shows the charts.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt

from src.data_collection import config
from src.data_collection.saver import save_processed
from src.var.historical import calculate_historical_var, calculate_historical_var_multi, rolling_historical_var
from src.var.expected_shortfall import calculate_expected_shortfall, rolling_expected_shortfall
from src.volatility.ewma import ewma_volatility, ewma_scaled_returns, ewma_scaled_historical_var
from src.visualization import charts

## Load Phase 3 output (portfolio returns)

In [ ]:
portfolio_returns = pd.read_csv(
    config.DATA_PROCESSED_DIR / 'portfolio_returns_phase3.csv',
    index_col=0, parse_dates=True
).iloc[:, 0]
portfolio_returns.tail()

## Phase 4 — Historical Simulation VaR

In [ ]:
# VaR summary table across all required windows and confidence levels
var_table = calculate_historical_var_multi(
    portfolio_returns,
    windows=[250, 500, 750],
    confidence_levels=[0.95, 0.99, 0.995],
)
var_table

In [ ]:
# Expected Shortfall at each confidence level (using full history)
es_by_confidence = {cl: calculate_expected_shortfall(portfolio_returns, cl) for cl in [0.95, 0.99, 0.995]}
var_by_confidence = {cl: calculate_historical_var(portfolio_returns, cl) for cl in [0.95, 0.99, 0.995]}
print('VaR:', var_by_confidence)
print('ES: ', es_by_confidence)

In [ ]:
# Rolling 250-day VaR time series
rolling_var_250 = rolling_historical_var(portfolio_returns, window=250, confidence_level=0.95)
rolling_var_250.tail()

In [ ]:
# Charts: return distribution, rolling VaR, tail loss, VaR vs ES
var_95 = var_by_confidence[0.95]
es_95 = es_by_confidence[0.95]

fig1 = charts.plot_return_distribution(portfolio_returns, var_95, es_95, confidence_level=0.95)
plt.show()

In [ ]:
fig2 = charts.plot_rolling_var(rolling_var_250, portfolio_returns)
plt.show()

In [ ]:
fig3 = charts.plot_tail_loss(portfolio_returns, var_95)
plt.show()

In [ ]:
fig4 = charts.plot_var_vs_es(var_by_confidence, es_by_confidence)
plt.show()

## Phase 5 — EWMA Volatility Scaling

In [ ]:
ewma_vol = ewma_volatility(portfolio_returns, lambda_=0.94)
ewma_vol.tail()

In [ ]:
scaled_returns = ewma_scaled_returns(portfolio_returns, lambda_=0.94)
scaled_returns.tail()

In [ ]:
ewma_var_95 = ewma_scaled_historical_var(portfolio_returns, confidence_level=0.95, lambda_=0.94)
print(f'Plain Historical VaR (95%): {var_95:.4f}')
print(f'EWMA-scaled VaR (95%):      {ewma_var_95:.4f}')

In [ ]:
fig5 = charts.plot_ewma_volatility(ewma_vol)
plt.show()

In [ ]:
fig6 = charts.plot_raw_vs_scaled_returns(portfolio_returns, scaled_returns)
plt.show()

In [ ]:
fig7 = charts.plot_var_comparison(var_95, ewma_var_95, confidence_level=0.95)
plt.show()

In [ ]:
fig8 = charts.plot_exception_comparison(portfolio_returns, var_95, ewma_var_95)
plt.show()

## Save Phase 4 & 5 outputs

In [ ]:
save_processed(var_table, 'var_summary_table.csv', config.DATA_PROCESSED_DIR)
save_processed(rolling_var_250.to_frame(), 'rolling_var_250d_95pct.csv', config.DATA_PROCESSED_DIR)
save_processed(ewma_vol.to_frame(), 'ewma_volatility.csv', config.DATA_PROCESSED_DIR)
save_processed(scaled_returns.to_frame(), 'ewma_scaled_returns.csv', config.DATA_PROCESSED_DIR)
print('Phase 4 & 5 outputs saved.')